### This is a Universal test file. Every function we change or make, needs to be tested here. Also the utility of each file can be found here

In [1]:
import os

# Some necessary directories for storing artifacts and message stores. Example data used in testing.

os.makedirs('artifacts', exist_ok=True)
os.makedirs('message_store', exist_ok=True)
os.makedirs("example_data", exist_ok=True)

#### Test for llms.py file
* You will need ollama to run opensource llms. Download here : https://ollama.com/
* install llama3.2 to test the examples
* Test to check that simple openai messages and chatollama messages works
* Test to check structured format works with both platforms
* Test to check tool calling is working with openai

In [ ]:
# All imports here

from utils.llms import *
from pydantic import BaseModel,Field
from utils.keys import set_api_keys
import json  # Added import for json usage
set_api_keys()

Openai key set successfully


In [7]:
# Test 1 : Is simple messaging working

# messages we are assuming openai style. Will be auto updated to langchain style. we will try both with only user and system
messages_test_1 = [{"role":"user", "content":"Hello How are you"}]
messages_test_2 = [{"role":"system","content":"Respond in victorian english"},{"role":"user", "content":"Hello How are you"}]

# responses from openai
response_openai_non_thinking = run_llm("gpt-4o-mini",messages_test_1,schema=None,tools=None)
response_openai_thinking = run_llm("o1-mini",messages_test_1,schema=None,tools=None)
response_openai_non_thinking_2 = run_llm("gpt-4o",messages_test_2,schema=None,tools=None)
response_openai_thinking_2 = run_llm("gpt-4.1-mini",messages_test_2,schema=None,tools=None)

print("Openai Runs completed")

# responses from opensource here llama2
response_open_source_non_thinking = run_llm("llama3.2",messages_test_1,schema=None,tools=None)
response_open_source_non_thinking_2 = run_llm("llama3.2",messages_test_2,schema=None,tools=None)

print("Llama2 runs completed")

print("Response from openai non thinking 1", response_openai_non_thinking)
print("Response from openai non thinking 2", response_openai_non_thinking_2)
print("Response from openai thinking 1", response_openai_thinking)
print("Response from openai thinking 2", response_openai_thinking_2)
print("Response from open source non thinking 1", response_open_source_non_thinking)
print("Response from open source non thinking 2", response_open_source_non_thinking_2)
   

Openai Runs completed
Llama2 runs completed
Response from openai non thinking 1 Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?
Response from openai non thinking 2 Good day to you! I find myself in fine spirits, thank you kindly for inquiring. And how might you be faring on this pleasant day?
Response from openai thinking 1 Hello! I'm doing well, thank you for asking. How can I assist you today?
Response from openai thinking 2 Good morrow to thee! I trust this missive findeth thee in fine fettle. How doth thy day fare?
Response from open source non thinking 1 I'm just a language model, so I don't have emotions or feelings like humans do. However, I'm functioning properly and ready to assist you with any questions or tasks you may have! How can I help you today?
Response from open source non thinking 2 Dear compatriot, I daresay I am in a state of optimal felicity, thank you for inquiring after my well-being. The gentl

In [8]:
# Test 2 : Is structured schema working. I will make a test schema to test it. 

class TestSchema(BaseModel):
    bengali : str = Field("Translate to bengali") # add common languages to test further
    hindi : str = Field("Translate to hindi")

messages = [{"role":"system", "content":"given the text translate it to bengali and hindi"},{"role":"user", "content":"Peter piper picked a bunch of pickled peppers"}]

# responses from openai
response_openai_non_thinking = run_llm("gpt-4o-mini",messages,schema=TestSchema,tools=None)
response_openai_thinking = run_llm("o1-mini",messages,schema=TestSchema,tools=None)

print("Openai Runs completed")

# responses from opensource here llama2
response_open_source_non_thinking = run_llm("llama3.2",messages,schema=TestSchema,tools=None)

print("Llama2 runs completed")

print("Response from openai non thinking", response_openai_non_thinking)
print("Response from openai thinking", response_openai_thinking)
print("Response from open source non thinking", response_open_source_non_thinking)


c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\llms.py:118: UserWarning: NOTE : Openai reasoning models do not support structured output so will  provide the structure in prompt and try. Susceptible to failure
  warnings.warn("NOTE : Openai reasoning models do not support structured output so will  provide the structure in prompt and try. Susceptible to failure",UserWarning)


Openai Runs completed
Llama2 runs completed
Response from openai non thinking bengali='পিটার পাইপার একটি ঝুড়ি আচার করা মরিচ তুলেছিল।' hindi='पीटर पाइपर ने अचार वाले मिर्चों का एक गुच्छा उठाया।'
Response from openai thinking bengali='পিটার পাইপার অনেকগুলো আচার করা মরিচ সংগ্রহ করেছিলেন' hindi='पीटर पाइपर ने अचार किए हुए मिर्चों का एक गुच्छा चुना'
Response from open source non thinking bengali='পিটার পাইপার একটি বন্ধু গজের মশলা আঁচড়ে আঁচড়ে পেপার' hindi='पीटर पाइपर ने एक बूंद की मिश्रित सब्जियों का संग्रह किया'


In [9]:
# Test 3 tool use only openai models supported here, While llama models can call tools, using it for NGD is not feasible untill we can accomodate slightly higher model sizes
# Here I will make a custom tool. Get the inputs from the LLM and then run the tool and given the outputs again to demonstrate a simple use

# simple substraction tool
def substract_two_numbers(a:float, b:float):
    return a-b

# simple addition tool
def add_two_numbers(a:float,b:float):
    return a+b

# Here I will make the tool description. For OS NGD will need to make a single function and describe the tool as below
tool_definition = [
    {
        "type":"function",
        "name":"substract_two_numbers",
        "description":"Simple tool to substract two numbers. Takes a and b as input calculates a-b",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"first number input"
                },
                "b":{
                    "type":"number",
                    "description":"second number input"
                }
            }
        }
    },

    {
        "type":"function",
        "name":"add_two_numbers",
        "description":"Simple tool to add two numbers. Takes a and b as input calculates a+b",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"first number input"
                },
                "b":{
                    "type":"number",
                    "description":"second number input"
                }
            }
        }
    },]

# Now we make the messages

messages = [{"role":"system","content":"you have 2 tools one to add numbers and another to substract. Use it to solve user equation. Multiple tool calls can be required to solve 1 equation"},
            {"role":"user","content":"solve: (8.95-6.71)"},
            {"role":"user","content":"solve: (8.95+6.71)"},]

response_openai_non_thinking = run_llm("gpt-4o-mini",messages,schema=None,tools=tool_definition)

# I would have written a custom execute tool code but will keep it pending now. Sometimes tool calls are supposed to be dependent on each other so I will write the code here and we can transfer to a function later
# We can multithread the above code no problem at all but maybe later
attempts = 0
while(response_openai_non_thinking.output[-1].type=="function_call" and attempts < 4):
    fn_calls = [i for i in response_openai_non_thinking.output if i.type=="function_call"]

    for call in fn_calls:
        messages.append(call)
        args = json.loads(call.arguments)

        if call.name == "substract_two_numbers":
            result = substract_two_numbers(args["a"],args["b"])
            messages.append({"type":"function_call_output", "call_id":call.call_id,"output":str(result)})
        
        if call.name == "add_two_numbers":
            result = add_two_numbers(args["a"],args["b"])
            messages.append({"type":"function_call_output", "call_id":call.call_id,"output":str(result)})
    response_openai_non_thinking = run_llm("gpt-4o-mini",messages,schema=None,tools=tool_definition)
    # No infinite loops for llms
    attempts = attempts + 1

print(response_openai_non_thinking.output_text)

The results are:

- \( 8.95 - 6.71 = 2.24 \)
- \( 8.95 + 6.71 = 15.66 \)


In [10]:
# Sometimes we may want to execute tools sequentially as an example ((6.45-2.34)+5.67) so first substract then add


messages = [{"role":"system","content":"you have 2 tools one to add numbers and another to substract. Use it to solve user equation. Multiple tool calls can be required to solve 1 equation"},
            {"role":"user","content":"solve: ((6.45-2.34)+5.67)"}]

response_openai_non_thinking = run_llm("gpt-4o-mini",messages,schema=None,tools=tool_definition,additional_args={"temperature":0,"max_tokens":2048,"parallel_tool_calls":False})

attempts = 0
while(response_openai_non_thinking.output[-1].type=="function_call" and attempts < 4):
    fn_calls = [i for i in response_openai_non_thinking.output if i.type=="function_call"]
    # I am printing this to verify sequential execution is this is printed twice the 2 tools are called not in parallel but sequentially
    print(fn_calls)

    for call in fn_calls:
        messages.append(call)
        args = json.loads(call.arguments)

        if call.name == "substract_two_numbers":
            result = substract_two_numbers(args["a"],args["b"])
            messages.append({"type":"function_call_output", "call_id":call.call_id,"output":str(result)})
        
        if call.name == "add_two_numbers":
            result = add_two_numbers(args["a"],args["b"])
            messages.append({"type":"function_call_output", "call_id":call.call_id,"output":str(result)})
    response_openai_non_thinking = run_llm("gpt-4o-mini",messages,schema=None,tools=tool_definition)
    # No infinite loops for llms
    attempts = attempts + 1

print(response_openai_non_thinking.output_text)

[ResponseFunctionToolCall(arguments='{"a":6.45,"b":2.34}', call_id='call_Ub02l3yaN3UIyuzPEYhSJqb4', name='substract_two_numbers', type='function_call', id='fc_05fcce5257eb8efe0068f267cc55bc8196ae39e953171763bc', status='completed')]
[ResponseFunctionToolCall(arguments='{"a":4.11,"b":5.67}', call_id='call_WAmDfaupA0gOxvWOGzOAGCg1', name='add_two_numbers', type='function_call', id='fc_05fcce5257eb8efe0068f267cf03188196bb6ff9348a248561', status='completed')]
The solution to the equation \(((6.45 - 2.34) + 5.67)\) is approximately \(9.78\).


#### Test for a2a framework for creating agents and testing multiagent collaboration

* We will first initialise some agents and test them indivisually like we did for run_llms
* Next we will test 3 agent collaboration. Simple test one will be host agent, one will be addition agent and one will be substraction agent

In [2]:
# All our imports go here

from utils.tool_definitions import send_message_definitions
from pydantic import BaseModel,Field
from utils.keys import set_api_keys
from utils.tools import send_message
from utils.tool_definitions import send_message_definitions
from a2a.Agent import Agent
from a2a.AgentCard import AgentCard
# We set the api keys as earlier
set_api_keys()

Openai key set successfully


In [5]:
# Test 1 initialise an agent with a closed llm and an open llm and try structured response. We will use previous schema 

class TestSchema(BaseModel):
    bengali : str = Field("Translate to bengali") # add common languages to test further
    hindi : str = Field("Translate to hindi")

# We will reuse this for both 
agent_details = AgentCard(agent_name="translate_to_languages", 
                                 agent_description="It is a useful agent that can translate from any language to hindi and bengali", 
                                 capabilities=["given an input string, it will translate to hindi and bengali"], 
                                 input_modes=["str"],
                                 output_modes=["a schema containing translated bengali and english"])

# Again we will use the same messages for both agents
messages = [{"role":"system", "content":"given the text translate it to bengali and hindi"},{"role":"user", "content":"Peter piper picked a bunch of pickled peppers"}]


agent_closed = Agent(agent_details, "gpt-4o-mini", TestSchema, None, None, None)
agent_closed_output = agent_closed.run_agent(messages)
print("Running agent closed completed")

agent_open = Agent(agent_details, "llama3.2", TestSchema, None, None, None)
agent_open_output = agent_open.run_agent(messages)
print("Running agent open completed")


print("Output of agent closed :", agent_closed_output)
print("Output of agent open :", agent_open_output)



Running agent closed completed
Running agent open completed
Output of agent closed : bengali='পিটার পাইপার একটি ঝুড়ি আচার করা মরিচ তুলেছিল।' hindi='पीटर पाइपर ने अचार वाले मिर्चों का एक गुच्छा उठाया।'
Output of agent open : bengali='পিটার পাইপার একটি ব্যাচ অপেক্ষা করেছিলেন মরিচ পেঁচো' hindi='पीटर पाइपर ने एक बैच की मिर्च को पकड़ा'


In [4]:
# Test 2 : Now we will test the tool capabilities of the agents so we will reuse the add and substract tools


# simple substraction tool
def substract_two_numbers(a:float, b:float):
    return a-b

# simple addition tool
def add_two_numbers(a:float,b:float):
    return a+b

# Here I will make the tool description. For OS NGD will need to make a single function and describe the tool as below
tool_definition = [
    {
        "type":"function",
        "name":"substract_two_numbers",
        "description":"Simple tool to substract two numbers. Takes a and b as input calculates a-b",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"first number input"
                },
                "b":{
                    "type":"number",
                    "description":"second number input"
                }
            }
        }
    },

    {
        "type":"function",
        "name":"add_two_numbers",
        "description":"Simple tool to add two numbers. Takes a and b as input calculates a+b",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"first number input"
                },
                "b":{
                    "type":"number",
                    "description":"second number input"
                }
            }
        }
    },]

# Now we make the messages

messages = [{"role":"system","content":"you have 2 tools one to add numbers and another to substract. Use it to solve user equation. Multiple tool calls can be required to solve 1 equation"},
            {"role":"user","content":"solve: (8.95-6.71)"},
            {"role":"user","content":"solve: (8.95+6.71)"},]


# Note only closed source llm here

agent_details = AgentCard(agent_name="addition_and_substraction_agent", 
                                 agent_description="It is a useful agent that can solves equations that does addition, substration or both", 
                                 capabilities=["given an equation that requires add or sub operations, it will add or substract them and solve them",
                                  "It has 2 tools 1 add tool and 1 substract tool"], 
                                 input_modes=["str"],
                                 output_modes=["The answer of the equation"])

tools = {"add_two_numbers":add_two_numbers,"substract_two_numbers":substract_two_numbers}
# Initialise the agent
agent = Agent(agent_details,"gpt-4o-mini", None,tools,tool_definition,additional_args={"parallel_tool_calls":False},available_agents=None,artifacts_req=None)

output_agent = agent.run_agent(messages)

print("ouput of agent :", output_agent)



Calling tool substract_two_numbers with args : {'a': 8.95, 'b': 6.71}
Calling tool add_two_numbers with args : {'a': 8.95, 'b': 6.71}
ouput of agent : ('The solutions are:\n\n- \\( 8.95 - 6.71 = 2.24 \\)\n- \\( 8.95 + 6.71 = 15.66 \\)', None)


In [3]:
# Test 3 : Now we will try to demonstrate agent communication. Here is the situation overall
# We will have a client agent and two other agents one which can add numbers and one which can substract numbers.
# Given a use query the client needs to delegate task to either 1 or both the agents

# simple substraction tool
def substract_two_numbers(a:float, b:float):
    return a-b

# simple addition tool
def add_two_numbers(a:float,b:float):
    return a+b




# so here we say there are 2 agents with 1 tool each, add or sub
tool_definition_agent_1 = [
    {
        "type":"function",
        "name":"substract_two_numbers",
        "description":"Simple tool to substract two numbers. Takes a and b as input calculates a-b",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"first number input"
                },
                "b":{
                    "type":"number",
                    "description":"second number input"
                }
            }
        }
    }]

tool_definition_agent_2 =   [{
        "type":"function",
        "name":"add_two_numbers",
        "description":"Simple tool to add two numbers. Takes a and b as input calculates a+b",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"first number input"
                },
                "b":{
                    "type":"number",
                    "description":"second number input"
                }
            }
        }
    },]


# based on that we create tools for both agents

tools_agent_1 = {"substract_two_numbers":substract_two_numbers}
tools_agent_2 = {"add_two_numbers":add_two_numbers}
tools_host_agent = {"send_message":send_message}


# Now we make our Agent cards

agent_1_details = AgentCard(agent_name="substraction_agent", 
                                 agent_description="It is a useful agent that can solves equations that does substraction operations only", 
                                 capabilities=["given an equation that requires sub operations, it will substract them and solve them",
                                  "It has 1 substract tool"], 
                                 input_modes=["str"],
                                 output_modes=["The answer of the substraction equation"])


agent_2_details = AgentCard(agent_name="addition_agent", 
                                 agent_description="It is a useful agent that can solves equations that does addition operations only", 
                                 capabilities=["given an equation that requires add operations, it will add them and solve them",
                                  "It has 1 addition tool"], 
                                 input_modes=["str"],
                                 output_modes=["The answer of the addition equation"])

host_agent_details = AgentCard(agent_name="Helpful_agent",
                               agent_description="Helpful user agent that communicates with other agents",
                               capabilities=["Communicate with other agents to solve an eq"],
                               input_modes=["str"],
                               output_modes=["answer to the query"])


# Now we initialise our agents

agent_1 = Agent(agent_1_details,"gpt-4o-mini",None,tools_agent_1,tool_definition_agent_1)
agent_2 = Agent(agent_2_details,"gpt-4o-mini",None,tools_agent_2,tool_definition_agent_2)

# For the client we will give the client the send_message tool to communicate with other agents
client_agent = Agent(host_agent_details,"gpt-4o-mini",None,tools_host_agent,[send_message_definitions],additional_args={"parallel_tool_calls":False},available_agents=[agent_1,agent_2],artifacts_req=None)


# We now make our messages and inform the llm about the existence of other agents

messages = [{"role":"system","content":f"""You will solve simple addition and substration operations the user give you. You are coordinating with 2 agents who can do this \
             for you. The agent details are given below. \
                1. Use the send_message tool to coordinate with the agents \
                2. Tell the agents clear task instructions \
             
             Agents available : {[agent_1_details.model_dump(), agent_2_details.model_dump()]}"""},
            {"role":"user","content":"solve: (8.95-6.71)"}, # call sub agent
            {"role":"user","content":"solve: (8.95+6.71)"}, # call add agent
            {"role":"user","content":"solve: (8.60+6.71) - 0.95"} ] # call both in correct seq


client_agent.run_agent(messages)

Calling tool send_message with args : {'target': 'substraction_agent', 'task_description': 'Solve the subtraction equation: (8.95 - 6.71).'}
Calling tool substract_two_numbers with args : {'a': 8.95, 'b': 6.71}
Calling tool send_message with args : {'target': 'addition_agent', 'task_description': 'Solve the addition equation: (8.95 + 6.71).'}
Calling tool add_two_numbers with args : {'a': 8.95, 'b': 6.71}
Calling tool send_message with args : {'target': 'addition_agent', 'task_description': 'Solve the addition equation: (8.60 + 6.71).'}
Calling tool add_two_numbers with args : {'a': 8.6, 'b': 6.71}
Calling tool send_message with args : {'target': 'substraction_agent', 'task_description': 'Solve the subtraction equation: (15.31 - 0.95).'}
Calling tool substract_two_numbers with args : {'a': 15.31, 'b': 0.95}


('Here are the results of the calculations:\n\n1. \\( 8.95 - 6.71 = 2.24 \\)\n2. \\( 8.95 + 6.71 = 15.66 \\)\n3. \\( 8.60 + 6.71 = 15.31 \\)\n4. \\( 15.31 - 0.95 = 14.36 \\)\n\nIf you need any further calculations, feel free to ask!',
 None)

### Test for Artifact system. We have made a rudimentary Artifact system. Here we will test them. Ideally artifacts are required for coding agents if they want to return the data 

* Movies data : https://www.kaggle.com/datasets/ahmedosamamath/imdb-dataset 

In [1]:
# Again let us import all required modules
from utils.tool_definitions import send_message_definitions
from pydantic import BaseModel,Field
from utils.keys import set_api_keys
from utils.tool_definitions import send_message_definitions, code_executor_definition, code_metadata_generator_definition
from a2a.Agent import Agent
from a2a.AgentCard import AgentCard
from a2a.Artifact import Artifact
import pandas as pd
import joblib
# We set the api keys as earlier
from utils.tools import send_message, code_executor, generate_metadata_for_artifacts
from utils.prompt_templates import generic_coding_agent_template
set_api_keys()

Openai key set successfully


In [2]:
# For the first test we will keep things simple. We will make an agent that will perform analysis on a simple data set lets say a csv file with some data

# lets get the data first. It is basically a movies data set with ratings and other info
df = pd.read_csv("example_data/movies_dataset.csv")
df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross
0,Blood Red Sky,(2021),"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN
1,Masters of the Universe: Revelation,(2021– ),"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN
2,The Walking Dead,(2010–2022),"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN
3,Rick and Morty,(2013– ),"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN
4,Army of Thieves,(2021),"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN


In [4]:
# Now let us initialise our agent

coding_agent_details = AgentCard(agent_name="data_analysis_agent",
                                    agent_description="It is a useful agent that can perform data analysis on a given data set and provide insights",
                                    capabilities=["Given a dataset, it can perform various data analysis tasks like summarization, statistical analysis, visualization etc."],
                                    input_modes=["task in str with the filenames provided to analyse data"],
                                    output_modes=["str","list out outputs including summaries, artifacts like plots and data files and their name and description"])

# Since this agent is not connected to a framework we will make an artifact and put it in artifacts folder

data_artifact = Artifact(name="movies_dataset",
                         description="A csv file containing movies data with ratings and other info",
                            data = df)
# We now save the artifact to artifacts folder
joblib.dump(data_artifact, "artifacts/movies_dataset.pkl")

# Now we make our agent with code execution and metadata generation tools
tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]

# We initialise the agent with the data artifact requirement and parallel tool calls disabled
coding_agent = Agent(coding_agent_details,"gpt-4o-mini",None,tools,tool_definitions,additional_args={"parallel_tool_calls":False},available_agents=None,artifacts_req=[data_artifact])

# Now we make our messages. Since we are providing the artifact req the Agent will be informed about the artifact in the run_agent function 
messages = [{"role":"system","content":generic_coding_agent_template},
            {"role":"user","content":"How many movies with genre as horror are there in the data set t?"}]

coding_agent_output = coding_agent.run_agent(messages)
print("Output of coding agent :", coding_agent_output)

Calling tool generate_metadata_for_artifacts with args : {'artifact_names': ['movies_dataset']}
artifacts loaded for metadata generation ['movies_dataset']
Calling tool code_executor with args : {'code': "import pandas as pd\n\ndef count_horror_movies(data):\n    # Concatenate all dataframes in the list into a single dataframe\n    df = pd.concat(data, ignore_index=True)\n    \n    # Filter for horror genre\n    horror_movies = df[df['GENRE'].str.contains('Horror', na=False)]\n    \n    # Count the number of horror movies\n    count = horror_movies.shape[0]\n    \n    return [f'There are {count} horror movies in the dataset.', 'horror_movies_count', 'A count of horror movies in the dataset', count]  \n\n", 'artifact_names': ['movies_dataset']}
function name count_horror_movies
code executed, output generated ['There are 553 horror movies in the dataset.', 'horror_movies_count', 'A count of horror movies in the dataset', 553]
Output of coding agent : ('The analysis reveals that there ar

In [3]:
# Now that a single agent with data provided works we will try with multiple agents.
# The setup is as follows:
# There are two agents 1. One is host and another is coding agent with data analysis capabilities
# We will make the host agent delegate tasks to the coding agent, the host agent will have a tool which tells it what data is available (similar to OS where the tool will return the data sets available)


# First we make the coding agent same as earlier
coding_agent_details = AgentCard(agent_name="data_analysis_agent",
                                    agent_description="It is a useful agent that can perform data analysis on a given data set and provide insights",
                                    capabilities=["Given a dataset, it can perform various data analysis tasks like summarization, statistical analysis, visualization etc."],
                                    input_modes=["task in str with the filenames provided to analyse data"],
                                    output_modes=["str","list out outputs including summaries, artifacts like plots and data files and their name and description"])


tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]

# We initialise the agent now without the data artifact requirement and parallel tool calls disabled. This will make the agent search for the artifact when the host agent tells it to
coding_agent = Agent(coding_agent_details,"gpt-4o",None,tools,tool_definitions,additional_args={"parallel_tool_calls":False},available_agents=None,system_instruction=generic_coding_agent_template)

# Now we make the host agent

host_agent_details = AgentCard(agent_name="Helpful_agent",
                               agent_description="Helpful user agent that communicates with user and other agents",
                               capabilities=["Communicate with other agents to communicate data_analysis tasks"],
                               input_modes=["str"],
                               output_modes=["answer to the query"])

# We need to make the search data tool for the host agent, this will tell the host what artifacts are available and will be later replaced by OSNGD search tool

def search_data_artifacts():
    # For now we will return the movies dataset only
    return ["movies_dataset"]

search_data_artifacts_definition = {
    "type":"function",
    "name":"search_data_artifacts",
    "description":"Tool to search for available data artifacts. No input required. Returns a list of artifact names available for data analysis",
    "parameters":{
        "type":"object",
        "properties":{}
    }
}

tools_host = {"send_message":send_message, "search_data_artifacts":search_data_artifacts}

system_instruction_host = f"""You will solve simple data analysis queries that the user gives you. You can assign the task to a coding agent telling it what you need to analyse and what dataset you need to do \
             for you. The agent details are given below. \
                1. Use the send_message tool to coordinate with the agents \
                2. Use the search_data_artifacts tool to find what dataset is available \
                2. Tell the agents clear task instructions (what to analyse and what dataset name to use) \
             
             Agents available : {[coding_agent_details.model_dump()]}"""

# Host knows coding agent is available
client_agent = Agent(host_agent_details,"gpt-4o-mini",None,tools_host,[send_message_definitions,search_data_artifacts_definition],additional_args={"parallel_tool_calls":False},available_agents=[coding_agent],artifacts_req=None,system_instruction=system_instruction_host)

# Now we make our messages and inform the llm about the existence of other agents


messages = [{"role":"user","content":"Which genre has the overall highest rating ?"}, # call sub agent
             ] # call both in correct seq

client_agent.run_agent(messages)

Calling tool search_data_artifacts with args : {}
Calling tool send_message with args : {'target': 'data_analysis_agent', 'task_description': "Analyze the 'movies_dataset' to determine which genre has the overall highest rating. Provide a summary of the findings."}
Calling tool generate_metadata_for_artifacts with args : {'artifact_names': ['movies_dataset']}
artifacts loaded for metadata generation ['movies_dataset']
Calling tool code_executor with args : {'code': 'def analyze_highest_rated_genre(data):\n    import pandas as pd\n    \n    # Extract the dataframe\n    df = data[0]\n    \n    # Clean the \'GENRE\' column by removing leading/trailing whitespace and splitting by comma\n    df[\'GENRE\'] = df[\'GENRE\'].str.strip().str.split(\', \')\n    \n    # Explode the dataframe to have one genre per row\n    df_exploded = df.explode(\'GENRE\')\n    \n    # Group by \'GENRE\' and calculate the mean rating for each genre\n    genre_ratings = df_exploded.groupby(\'GENRE\')[\'RATING\'].m

('The genre with the highest average rating is **Animation** with a rating of **7.38**. \n\nAdditionally, a data artifact has been generated:\n\n- **Name:** highest_rated_genre\n- **Description:** Dataframe showing the genre with the highest average rating.',
 [[<a2a.Artifact.Artifact at 0x1ca62d93fd0>]])

### Testing of Message store. This is useful object that helps to differentiate between messages of different agents.
* Each Agent talks to another agent using tasks. Before a task is create a message object is created for a new conversation which is stored
* If the agent talks again to the same agent the previous history will be retrieved using the task id. The messages will be updated this time and then saved
* We do simple filtering but can do long and short term filtering
* Note every agent is initialised with its system prompt so we are saving a lot of space by not storing the system prompt in memory
* Also for multiple tasks sent in seq to the same agent. The previous message stores are deleted and the complete history is updated for the most recent task

In [3]:
# We will reuse the same example but run it using 2 questions to demonstrate message store working

# Now that a single agent with data provided works we will try with multiple agents.
# The setup is as follows:
# There are two agents 1. One is host and another is coding agent with data analysis capabilities
# We will make the host agent delegate tasks to the coding agent, the host agent will have a tool which tells it what data is available (similar to OS where the tool will return the data sets available)


# First we make the coding agent same as earlier
coding_agent_details = AgentCard(agent_name="data_analysis_agent",
                                    agent_description="It is a useful agent that can perform data analysis on a given data set and provide insights",
                                    capabilities=["Given a dataset, it can perform various data analysis tasks like summarization, statistical analysis, visualization etc."],
                                    input_modes=["task in str with the filenames provided to analyse data"],
                                    output_modes=["str","list out outputs including summaries, artifacts like plots and data files and their name and description"])


tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]

# We initialise the agent now without the data artifact requirement and parallel tool calls disabled. This will make the agent search for the artifact when the host agent tells it to
coding_agent = Agent(coding_agent_details,"gpt-4o",None,tools,tool_definitions,additional_args={"parallel_tool_calls":False},available_agents=None,system_instruction=generic_coding_agent_template)

# Now we make the host agent

host_agent_details = AgentCard(agent_name="Helpful_agent",
                               agent_description="Helpful user agent that communicates with user and other agents",
                               capabilities=["Communicate with other agents to communicate data_analysis tasks"],
                               input_modes=["str"],
                               output_modes=["answer to the query"])

# We need to make the search data tool for the host agent, this will tell the host what artifacts are available and will be later replaced by OSNGD search tool

def search_data_artifacts():
    # For now we will return the movies dataset only
    return ["movies_dataset"]

search_data_artifacts_definition = {
    "type":"function",
    "name":"search_data_artifacts",
    "description":"Tool to search for available data artifacts. No input required. Returns a list of artifact names available for data analysis",
    "parameters":{
        "type":"object",
        "properties":{}
    }
}

tools_host = {"send_message":send_message, "search_data_artifacts":search_data_artifacts}

system_instruction_host = f"""You will solve simple data analysis queries that the user gives you. You can assign the task to a coding agent telling it what you need to analyse and what dataset you need to do \
             for you. The agent details are given below. \
                1. Use the send_message tool to coordinate with the agents \
                2. Use the search_data_artifacts tool to find what dataset is available \
                2. Tell the agents clear task instructions (what to analyse and what dataset name to use) \
             
             Agents available : {[coding_agent_details.model_dump()]}"""

# Host knows coding agent is available
client_agent = Agent(host_agent_details,"gpt-4o-mini",None,tools_host,[send_message_definitions,search_data_artifacts_definition],additional_args={"parallel_tool_calls":False},available_agents=[coding_agent],artifacts_req=None,system_instruction=system_instruction_host)

# Now we make our messages and inform the llm about the existence of other agents


messages = [{"role":"user","content":"Which genre has the overall highest rating ?"},
            {"role":"user","content":"What is the average rating of comedy genre"} # call sub agent
             ] # call both in correct seq

client_agent.run_agent(messages)

Messages being sent to agent [{'role': 'user', 'content': "Analyze the 'movies_dataset' to find which genre has the overall highest rating."}]
artifacts loaded for metadata generation ['movies_dataset']
Messages being sent to agent [{'role': 'user', 'content': "Analyze the 'movies_dataset' to find which genre has the overall highest rating."}, {'role': 'assistant', 'content': "The genre with the highest average rating is **Animation** with a rating of **7.38**.Addtionally some data artifacts have been generated with names  ['highest_rated_genre'] and \n descriptions ['The genre with the highest average rating and its rating.']"}, {'role': 'user', 'content': "Calculate the average rating of the comedy genre using the 'movies_dataset'."}]
artifacts loaded for metadata generation ['movies_dataset']


('The genre with the overall highest rating is **Animation** with a rating of **7.38**. \n\nAdditionally, the average rating for the **Comedy** genre is **6.83**.',
 [[<a2a.Artifact.Artifact at 0x2115d0ce410>]])

### Test and Plan for OSNDG API KEY STATUS

In [1]:
# importing os library
from osdatahub import NGD, PlacesAPI, Extent, LinkedIdentifiersAPI
import geopandas as gpd
from utils.keys import set_api_keys
import os
set_api_keys()


Openai and ngd key set successfully


In [2]:
# We do not have access so just demonstrating code here
ngd = NGD(key=os.getenv("OSNDG_API_KEY"), collection="bld-fts-building-3")
area = ngd.query(max_results=5, cql_filter=f"description = 'Residential Building'")
buildings = gpd.GeoDataFrame.from_features(area["features"])


In [4]:
# Using Places API
places_api = PlacesAPI(key=os.getenv("OSNDG_API_KEY"))
address = places_api.find("kfc")
address = gpd.GeoDataFrame.from_features(address["features"])
address

,geometry,UPRN,UDPRN,ADDRESS,ORGANISATION_NAME,BUILDING_NAME,DEPENDENT_THOROUGHFARE_NAME,THOROUGHFARE_NAME,POST_TOWN,POSTCODE,...,PARISH_CODE,PARENT_UPRN,LAST_UPDATE_DATE,ENTRY_DATE,BLPU_STATE_DATE,LANGUAGE,MATCH,MATCH_DESCRIPTION,DELIVERY_POINT_SUFFIX,DEPENDENT_LOCALITY
0,POINT (317477 197654),43085413,28426641,"KFC, UNIT 1, NORTH COURT, HIGH STREET, BLACKWO...",KFC,UNIT 1,NORTH COURT,HIGH STREET,BLACKWOOD,NP12 1FG,...,W04000730,43085412,10/02/2016,18/08/2006,27/04/2010,EN,0.3,NO MATCH,1A,NaN
1,POINT (317477 197654),43085413,28426641,"KFC, UNIT 1, NORTH COURT, HIGH STREET, COED-DU...",KFC,UNIT 1,NORTH COURT,HIGH STREET,COED-DUON,NP12 1FG,...,W04000730,43085412,10/02/2016,18/08/2006,27/04/2010,CY,0.3,NO MATCH,1A,NaN
2,POINT (353006 116357),30127467,52635447,"KFC (GB) LTD, UNIT 8, WESTERN AVENUE, HOUNDSTO...",KFC (GB) LTD,UNIT 8,NaN,WESTERN AVENUE,YEOVIL,BA22 8YQ,...,E04008773,NaN,31/03/2023,07/02/2013,07/02/2013,EN,0.2,NO MATCH,1Q,HOUNDSTONE BUSINESS PARK


In [12]:
buildings

,geometry,osid,theme,isinsite,changetype,buildinguse,description,versiondate,connectivity,primarysiteid,...,buildinguse_addresscount_other,buildinguse_addresscount_total,constructionmaterial_updatedate,buildingage_thirdpartyprovenance,constructionmaterial_evidencedate,constructionmaterial_capturemethod,buildinguse_addresscount_commercial,buildinguse_addresscount_residential,basementpresence_thirdpartyprovenance,constructionmaterial_thirdpartyprovenance
0,"POLYGON ((-0.02507 51.13741, -0.02506 51.13744...",000014be-cd94-404d-91f2-aaa66dee6cf8,Buildings,True,New,Residential Accommodation,Residential Building,2024-07-20,Standalone,e8da182e-4a0c-42ef-bed1-714eff4ef92c,...,0,0,None,None,None,None,0,0,None,None
1,"POLYGON ((-0.75645 52.14902, -0.7564 52.14904,...",000023b4-7b71-4e2b-adcd-86bdb7bb3cd7,Buildings,True,Modified Attributes,Residential Accommodation,Residential Building,2025-05-29,Standalone,9112f0be-7d3d-4b9a-b807-dac46ff16b74,...,0,0,None,None,None,None,0,0,None,None
2,"POLYGON ((-2.56088 52.32852, -2.56085 52.32861...",00003b09-8fec-4a89-a638-a3603aa019f3,Buildings,True,New,Residential Accommodation,Residential Building,2024-07-20,Standalone,bef42611-328f-40a3-b967-1807dc0aec71,...,0,1,2024-07-19,Historic England (Listed Buildings),1952-10-06,Third Party Unknown,0,1,Historic England (Listed Buildings),Historic England (Listed Buildings)
3,"POLYGON ((-1.4272 53.0345, -1.42711 53.03451, ...",00003eba-3e1c-4bf6-8365-8e80e8145999,Buildings,True,Modified Attributes,Residential Accommodation,Residential Building,2025-09-24,Standalone,f3f2b75a-2115-4983-b3ba-30f9ae030ec4,...,0,1,2025-09-24,None,2025-09-19,Automated Process,0,1,None,Infill Modelling
4,"POLYGON ((-4.22049 51.76172, -4.22044 51.76178...",000067ef-2950-48f8-9133-2e7a7deb7584,Buildings,True,New,Residential Accommodation,Residential Building,2024-07-20,Standalone,e672958a-85b8-4cdd-8838-21b9f20b2f75,...,0,0,None,None,None,None,0,0,None,None


In [6]:
buildings.columns

Index(['geometry', 'osid', 'theme', 'isinsite', 'changetype', 'buildinguse',
       'description', 'versiondate', 'connectivity', 'primarysiteid',
       'sitereference', 'uprnreference', 'mainbuildingid', 'numberoffloors',
       'basementpresence', 'buildingage_year', 'geometry_area_m2',
       'buildingpartcount', 'buildingage_period', 'buildingage_source',
       'connectivity_count', 'containingsitecount', 'geometry_updatedate',
       'constructionmaterial', 'buildingpartreference',
       'numberoffloors_source', 'buildingage_updatedate',
       'buildinguse_updatedate', 'description_updatedate',
       'versionavailabletodate', 'basementpresence_source',
       'connectivity_updatedate', 'buildingage_evidencedate',
       'versionavailablefromdate', 'buildingage_capturemethod',
       'mainbuildingid_updatedate', 'numberoffloors_updatedate',
       'buildinguse_oslandusetiera', 'basementpresence_updatedate',
       'constructionmaterial_source', 'numberoffloors_evidencedate',
 

In [24]:
address.columns

Index(['geometry', 'UPRN', 'UDPRN', 'ADDRESS', 'ORGANISATION_NAME',
       'BUILDING_NAME', 'DEPENDENT_THOROUGHFARE_NAME', 'THOROUGHFARE_NAME',
       'POST_TOWN', 'POSTCODE', 'RPC', 'X_COORDINATE', 'Y_COORDINATE',
       'STATUS', 'LOGICAL_STATUS_CODE', 'CLASSIFICATION_CODE',
       'CLASSIFICATION_CODE_DESCRIPTION', 'LOCAL_CUSTODIAN_CODE',
       'LOCAL_CUSTODIAN_CODE_DESCRIPTION', 'COUNTRY_CODE',
       'COUNTRY_CODE_DESCRIPTION', 'POSTAL_ADDRESS_CODE',
       'POSTAL_ADDRESS_CODE_DESCRIPTION', 'BLPU_STATE_CODE',
       'BLPU_STATE_CODE_DESCRIPTION', 'TOPOGRAPHY_LAYER_TOID', 'WARD_CODE',
       'PARISH_CODE', 'PARENT_UPRN', 'LAST_UPDATE_DATE', 'ENTRY_DATE',
       'BLPU_STATE_DATE', 'LANGUAGE', 'MATCH', 'MATCH_DESCRIPTION',
       'DELIVERY_POINT_SUFFIX', 'DEPENDENT_LOCALITY'],
      dtype='object')

In [15]:
address.iloc[0].CLASSIFICATION_CODE_DESCRIPTION

'Restaurant / Cafeteria'

In [26]:
buildings.columns.intersection(address.columns)

Index(['geometry'], dtype='object')

### OS NGD Agentic Version Test 1
* Address, Buildings, named area is added
* geographical locations can be done by Named area, Counties need to be handled
* We will begin with very simple and a few queries

In [11]:
# First the imports and the keys
import warnings
warnings.filterwarnings("ignore")
from utils.keys import set_api_keys
import os
from utils.tool_definitions import send_message_definitions
from pydantic import BaseModel,Field
from utils.tool_definitions import *
from a2a.Agent import Agent
from utils.card_templates import *
from a2a.Artifact import Artifact
import pandas as pd
import joblib
# We set the api keys as earlier
from utils.tools import *
from utils.prompt_templates import *
set_api_keys()

In here
Openai and ngd key set successfully


In [2]:
# Ok Step 1 : Let us initialise our agents tools and utilities that the agents will use.


#coding agent
tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]

coding_agent = Agent(coding_agent_details,
                     "gpt-4.1",
                     None,
                     tools,
                     tool_definitions,
                     additional_args={"parallel_tool_calls":False},
                     available_agents=None,
                     system_instruction=generic_coding_agent_template)


#plotting agent
tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]
plotting_agent = Agent(plotting_agent_card,
                     "gpt-4.1",
                     None,
                     tools,
                     tool_definitions,
                     additional_args={"parallel_tool_calls":False},
                     available_agents=None,
                     system_instruction=plotting_agent_template)



# Address agent
address_agent_cards =[coding_agent_details.model_dump()]
address_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
address_system_prompt = places_prompt + f"\n <AVAILABLE AGENTS> {address_agent_cards} <AVAILABLE AGENTS>"

address_agent = Agent(agent_details=address_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=address_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=address_system_prompt)


# Buildings agent
buildings_agent_cards =[coding_agent_details.model_dump()]
buildings_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
buildings_system_prompt = buildings_prompt + f"\n <AVAILABLE AGENTS> {buildings_agent_cards} <AVAILABLE AGENTS>"

buildings_agent = Agent(agent_details=building_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=address_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=buildings_system_prompt)



# Named Area agent
named_area_agent_cards =[coding_agent_details.model_dump()]
named_area_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
named_area_system_prompt = named_area_prompt + f"\n <AVAILABLE AGENTS> {named_area_agent_tools} <AVAILABLE AGENTS>"

named_area_agent = Agent(agent_details=named_area_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=address_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=named_area_system_prompt)

# planning agent
planning_agent = Agent(agent_details=planning_agent_card,
                       llm_name = "gpt-4o-mini", schema=None,
                       tools=None,
                       tool_definitions=None,
                       additional_args=None,
                       available_agents=None,
                       artifacts_req=None,
                       system_instruction=planning_agent_prompt)

# host agent
host_agent_cards = [planning_agent_card.model_dump(),address_agent_card.model_dump(), named_area_agent_card.model_dump(),building_agent_card.model_dump(),plotting_agent_card.model_dump()]
host_agent_tools = {"send_message":send_message,'generate_metadata_for_all_artifacts':generate_metadata_for_all_artifacts}
host_tools_definitions = [send_message_definitions,metadata_all_artifacts]
host_system_message = host_prompt_template_for_os_version_2 + f"\n <AVAILABLE AGENTS> {host_agent_cards} <AVAILABLE AGENTS>"

host_agent = Agent(agent_details=host_agent_card,
                   llm_name="gpt-4.1",schema=None,
                   tools=host_agent_tools,
                   tool_definitions=host_tools_definitions,
                   additional_args={"parallel_tool_calls":False},
                   available_agents=[planning_agent,address_agent,named_area_agent,buildings_agent,plotting_agent],
                   artifacts_req=None,
                   system_instruction=host_system_message)


In [3]:
# All initialised lets try a single query and test. We definitely need more abstraction for the last cell.

messages = [{"role":"user","content":"Find hospitals in a 10km radius of Exeter cathedral."}]
host_agent.run_agent(messages)

Calling tool send_message with args : {'target': 'planning_agent', 'task_description': 'Provide a sequence of steps to find hospitals in a 10km radius of Exeter cathedral.'}
Messages sent to agent planning_agent [{'role': 'user', 'content': 'Provide a sequence of steps to find hospitals in a 10km radius of Exeter cathedral.'}]
Calling tool send_message with args : {'target': 'named_area', 'task_description': 'Find the named area polygon for Exeter.'}
Messages sent to agent named_area [{'role': 'user', 'content': 'Find the named area polygon for Exeter.'}]
Calling tool call_os_ngd with args : {'filters': None, 'bbox': None, 'polygon_or_point': True, 'street_address': None, 'filename': 'exeter_named_area'}
NGD query result {'name': 'exeter_named_area', 'description': 'A geopandas dataframe containing named area data with bbox applied as per user request.', 'count': 98}
Calling tool send_message with args : {'target': 'data_analysis_agent', 'task_description': "Filter the artifact 'exeter

Traceback (most recent call last):
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\a2a\SendMessage.py", line 27, in send_messages
    output,artifact = self.agent.run_agent(self.task.history)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\a2a\Agent.py", line 62, in run_agent
    messages,artifacts = self.run_tools(messages,fn_calls)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\a2a\Agent.py", line 110, in run_tools
    result = self.tools[call.name](**args)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\utils\tools.py", line 37, in call_os_ngd
    result = ngd_util_mapping[kwargs["ngd_name"]](**{k:v for k,v in kwargs.items() if(k!="ngd_name" and k in args) })
             ^^^^^^^^^^^^^^^^^^^^^^^

Calling tool call_os_ngd with args : {'filters': ['Place Of Worship'], 'bbox': 'exeter_named_area_filtered', 'polygon_or_point': True, 'street_address': False, 'filename': 'exeter_cathedral_search'}
NGD query result {'name': 'exeter_cathedral_search', 'description': 'A geopandas dataframe containing address data with filters and bbox applied as per user request.', 'count': 27}
Calling tool send_message with args : {'target': 'data_analysis_agent', 'task_description': "Filter the artifact 'exeter_cathedral_search' to find Exeter Cathedral specifically."}
Messages sent to agent data_analysis_agent [{'role': 'user', 'content': "Filter the artifact 'exeter_cathedral_search' to find Exeter Cathedral specifically."}]
Calling tool generate_metadata_for_artifacts with args : {'artifact_names': ['exeter_cathedral_search']}
artifacts loaded for metadata generation ['exeter_cathedral_search']
Calling tool code_executor with args : {'code': 'def filter_exeter_cathedral(data: list):\n    df = data[

('I have found 8 hospitals within a 10km radius of Exeter Cathedral. The results include:\n\n1. NORTHERN DEVON HEALTHCARE TRUST, FRANKLYN HOUSE, EXETER, EX2 9HS\n2. WONFORD HOUSE HOSPITAL, DRYDEN ROAD, EXETER, EX2 5AF\n3. EXETER AND MID DEVON HOSPITAL, CHURCH LANE, WONFORD, EXETER, EX2 5AD\n4. ROYAL DEVON AND EXETER HOSPITAL, BARRACK ROAD, EXETER, EX2 5DW\n5. THE EXETER NUFFIELD HOSPITAL, WONFORD ROAD, EXETER, EX2 4UG\n\nA map has been generated showing Exeter Cathedral, a 10km buffer, and all hospitals within that radius.\n\nIf you would like to view or download the map, let me know!',
 [<a2a.Artifact.Artifact at 0x25908757c50>])

### Preprocessing the OS NGD data
* Context : We have been provided the os ngd data and we need to preprocess it into a simple form
* Each data contains a zip containing the gpkg and a summary json
* There will be bunch of random code here for testing

In [2]:
# reading a sample file
import geopandas as gpd
import joblib
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Lets explore the address database and make decisions on what to return in a single api call

data_builtaddress = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\add_gb_builtaddress\add_gb_builtaddress.gpkg")
data_historicaddress = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\add_gb_historicaddress\add_gb_historicaddress.gpkg")
data_nonaddress = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\add_gb_nonaddressableobject\add_gb_nonaddressableobject.gpkg")
data_streetaddress = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\add_gb_streetaddress\add_gb_streetaddress.gpkg")


In [4]:
data_builtaddress.columns

Index(['uprn', 'versiondate', 'versionavailablefromdate',
       'versionavailabletodate', 'changetype', 'theme', 'description',
       'organisationname', 'poboxnumber', 'subname', 'name', 'number',
       'streetname', 'locality', 'townname', 'islandname', 'postcode',
       'fulladdress', 'country', 'alternatelanguagesubname',
       'alternatelanguagename', 'alternatelanguagenumber',
       'alternatelanguagestreetname', 'alternatelanguagelocality',
       'alternatelanguagetownname', 'alternatelanguageislandname',
       'alternatelanguage', 'alternatelanguagefulladdress', 'floorlevel',
       'lowestfloorlevel', 'highestfloorlevel', 'classificationcode',
       'classificationdescription', 'primaryclassificationdescription',
       'secondaryclassificationdescription',
       'tertiaryclassificationdescription',
       'quaternaryclassificationdescription', 'buildstatus', 'buildstatusdate',
       'addressstatus', 'postcodesource', 'parentuprn', 'rootuprn',
       'hierarchylevel

In [5]:
set(data_historicaddress["classificationdescription"].unique()) - set(data_builtaddress["classificationdescription"].unique()) 

{'Advertising Hoarding',
 'Agricultural - Applicable to land in farm ownership and not run as a separate business enterprise',
 'Automated Teller Machine (ATM)',
 'Broadcasting (TV / Radio)',
 'Bus Shelter',
 'Car / Coach / Commercial Vehicle / Taxi Parking / Park And Ride Site',
 'Electricity Sub-Station',
 'General Storage Land',
 'Hopper / Silo / Cistern / Tank',
 'House In Multiple Occupation',
 'Land',
 'Lock-Up Garage / Garage Court',
 'Mineral / Ore Working / Quarry / Mine',
 'Postal Box',
 'Residential',
 'Residential Institution',
 'Sheltered Accommodation',
 'Static Water',
 'Storage Land',
 'Street Record',
 'Telephone Box',
 'Unused Land'}

In [6]:
# From this built address, historic address and non addressable objects can be merged so search in all and merge
# Street Address is different
print(set(data_builtaddress.columns) - set(data_historicaddress.columns))
print(set(data_builtaddress.columns) - set(data_nonaddress.columns))
print(set(data_builtaddress.columns) - set(data_streetaddress.columns))

set()
set()
{'positionalaccuracy', 'usrnmatchindicator', 'islandname', 'buildstatusdate', 'lowestfloorlevel', 'northing', 'poboxnumber', 'secondaryclassificationdescription', 'addressstatus', 'localcustodiancode', 'postcodesource', 'primaryclassificationdescription', 'parentuprn', 'number', 'tertiaryclassificationdescription', 'hierarchylevel', 'longitude', 'localcustodiandescription', 'alternatelanguagesubname', 'name', 'alternatelanguageislandname', 'classificationcode', 'fulladdress', 'alternatelanguagenumber', 'alternatelanguagefulladdress', 'classificationdescription', 'rootuprn', 'alternatelanguagename', 'lowertierlocalauthoritygsscode', 'quaternaryclassificationdescription', 'latitude', 'floorlevel', 'subname', 'highestfloorlevel', 'buildstatus', 'organisationname', 'postcode', 'uprn', 'easting'}


In [7]:
# Lets explore buildings now
data_buildings = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\bld_fts_building\bld_fts_building.gpkg")
data_buildingsacces = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\bld_fts_buildingaccesslocation\bld_fts_buildingaccesslocation.gpkg")
data_buildingline = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\bld_fts_buildingline\bld_fts_buildingline.gpkg")
data_buildingpart = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\bld_fts_buildingpart\bld_fts_buildingpart.gpkg")



In [8]:
data_buildings.columns[data_buildings.columns.str.contains("part")]

Index(['buildingpartcount', 'constructionmaterial_thirdpartyprovenance',
       'buildingage_thirdpartyprovenance',
       'basementpresence_thirdpartyprovenance'],
      dtype='object')

In [9]:
# All different so need to explore this more
print(set(data_buildings.columns) - set(data_buildingsacces.columns))
print(set(data_buildings.columns) - set(data_buildingline.columns))
print(set(data_buildings.columns) - set(data_buildingpart.columns))


{'height_updatedate', 'roofshapeaspect_confidenceindicator', 'buildinguse', 'basementpresence_capturemethod', 'buildinguse_addresscount_other', 'buildinguse_addresscount_total', 'roofshapeaspect_updatedate', 'roofshapeaspect_areafacingsoutheast_m2', 'buildingpartcount', 'height_relativemax_m', 'mainbuildingid', 'roofmaterial_primarymaterial', 'basementpresence_evidencedate', 'basementpresence_source', 'roofshapeaspect_areafacingsouthwest_m2', 'connectivity_count', 'numberoffloors_evidencedate', 'buildinguse_oslandusetiera', 'buildingage_evidencedate', 'basementpresence_thirdpartyprovenance', 'buildingage_year', 'roofshapeaspect_areapitched_m2', 'roofshapeaspect_areaflat_m2', 'roofshapeaspect_capturemethod', 'buildinguse_updatedate', 'roofshapeaspect_areatotal_m2', 'roofshapeaspect_areafacingwest_m2', 'geometry_area_m2', 'height_absolutemax_m', 'roofmaterial_confidenceindicator', 'buildingage_source', 'height_relativeroofbase_m', 'roofshapeaspect_shape', 'constructionmaterial_capturemet

In [10]:
(set(data_buildingline["description"].unique())) - set(data_buildings["description"].unique())

{'Building Internal Division',
 'Building Occupier Division',
 'Overhanging Building Edge'}

In [11]:
data_buildings.attrs = {"name":"buildings"}

In [12]:
data_buildings.attrs

{'name': 'buildings'}

In [13]:
import inspect
from utils.os_utils_data_raw import query_address
args = inspect.signature(query_address).parameters
print(args.keys())



odict_keys(['filters', 'bbox', 'street_address', 'filename'])


In [14]:
from utils.prompt_templates import get_filterable_features

get_filterable_features("address")

In here


{'Activity / Leisure / Sports Centre',
 'Additional Mail / Packet Addressee',
 'Advertising Hoarding',
 'Agricultural',
 'Agricultural - Applicable to land in farm ownership and not run as a separate business enterprise',
 'Air Force',
 'Allotment',
 'Amenity - Open areas not attracting visitors',
 'Amusements',
 'Ancillary Building',
 'Animal Centre',
 'Animal Services',
 'Archaeological Dig Site',
 'Arena / Stadium',
 'Army',
 'Automated Teller Machine (ATM)',
 'Bank / Financial Service',
 'Betting Shop',
 'Bingo Hall / Cinema / Conference / Exhibition Centre / Theatre / Concert Hall',
 'Boarding / Guest House / Bed And Breakfast / Youth Hostel',
 'Brewery',
 'Broadcasting (TV / Radio)',
 "Builders' Yard",
 'Bus Shelter',
 'Butt / Hide',
 'Car / Coach / Commercial Vehicle / Taxi Parking / Park And Ride Site',
 'Caravan',
 'Care / Nursing Home',
 'Castle / Historic Ruin',
 'Cemetery',
 'Cemetery / Crematorium / Graveyard. In Current Use.',
 'Central Government Service',
 'Channel / Co

In [26]:
data_water = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\wtr_fts_water\wtr_fts_water.gpkg")
data_waterpoint = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\wtr_fts_waterpoint\wtr_fts_waterpoint.gpkg")
data_waterlink = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\wtr_ntwk_waterlink\wtr_ntwk_waterlink.gpkg")
data_waterlinkset = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\wtr_ntwk_waterlinkset\wtr_ntwk_waterlinkset.gpkg")


In [16]:
print(set(data_water.columns) - set(data_waterpoint.columns))
print(set(data_water.columns) - set(data_waterlink.columns))
print(set(data_water.columns) - set(data_waterlinkset.columns))
print(set(data_waterlinkset.columns) - set(data_waterlink.columns))

{'status_updatedate', 'oslanduse_evidencedate', 'oslanduse_capturemethod', 'containingsitecount', 'smallestsite_landusetiera', 'oslandusetiera', 'oslandcovertierb', 'description_capturemethod', 'oslandcover_capturemethod', 'status', 'nlud_code', 'oslandcovertiera', 'smallestsite_siteid', 'lowertierlocalauthority_gsscode', 'smallestsite_landusetierb', 'geometry_capturemethod', 'oslanduse_updatedate', 'address_secondarydescription', 'watertype', 'lowertierlocalauthority_count', 'oslandusetierb', 'geometry_area_m2', 'largestsite_landusetierb', 'oslandcover_updatedate', 'nlud_orderdescription', 'address_primarydescription', 'largestsite_landusetiera', 'oslandcover_evidencedate', 'address_classificationcode', 'associatedstructure', 'nlud_groupdescription'}
{'status_updatedate', 'oslanduse_evidencedate', 'oslanduse_capturemethod', 'containingsitecount', 'smallestsite_landusetiera', 'oslandusetiera', 'oslandcovertierb', 'description_capturemethod', 'oslandcover_capturemethod', 'status', 'nlud

In [59]:
data_water.columns

Index(['osid', 'toid', 'versiondate', 'versionavailablefromdate',
       'versionavailabletodate', 'firstdigitalcapturedate', 'changetype',
       'geometry_area_m2', 'geometry_evidencedate', 'geometry_updatedate',
       'geometry_capturemethod', 'theme', 'description',
       'description_evidencedate', 'description_updatedate',
       'description_capturemethod', 'oslandcovertiera', 'oslandcovertierb',
       'oslandcover_evidencedate', 'oslandcover_updatedate',
       'oslandcover_capturemethod', 'oslandusetiera', 'oslandusetierb',
       'oslanduse_evidencedate', 'oslanduse_updatedate',
       'oslanduse_capturemethod', 'watertype', 'associatedstructure',
       'isobscured', 'physicallevel', 'capturespecification',
       'containingsitecount', 'smallestsite_siteid',
       'smallestsite_landusetiera', 'smallestsite_landusetierb',
       'largestsite_landusetiera', 'largestsite_landusetierb', 'nlud_code',
       'nlud_orderdescription', 'nlud_groupdescription',
       'address_cl

In [58]:
data_waterpoint.columns

Index(['osid', 'toid', 'versiondate', 'versionavailablefromdate',
       'versionavailabletodate', 'firstdigitalcapturedate', 'changetype',
       'geometry_evidencedate', 'geometry_updatedate', 'geometry_source',
       'theme', 'description', 'description_evidencedate',
       'description_updatedate', 'description_source', 'name1_text',
       'name1_language', 'name1_evidencedate', 'name1_updatedate',
       'name1_source', 'name2_text', 'name2_language', 'name2_evidencedate',
       'name2_updatedate', 'name2_source', 'operationalstatus', 'isobscured',
       'physicallevel', 'capturespecification', 'geometry'],
      dtype='object')

In [17]:
data_waterlink.columns

Index(['osid', 'toid', 'versiondate', 'versionavailablefromdate',
       'versionavailabletodate', 'changetype', 'geometry_length_m',
       'geometry_evidencedate', 'geometry_updatedate', 'geometry_source',
       'theme', 'description', 'description_evidencedate',
       'description_updatedate', 'description_source', 'primacy', 'watertype',
       'physicallevel', 'physicalcontainment', 'flowdirection', 'permanence',
       'capturespecification', 'levelofdetail', 'catchmentname', 'catchmentid',
       'width_average', 'width_minimum', 'width_maximum',
       'width_derivationmethod', 'width_evidencedate', 'width_updatedate',
       'gradient', 'startnode', 'endnode', 'nameid', 'name1_text',
       'name1_language', 'name2_text', 'name2_language', 'namesecondaryid',
       'namesecondary1_text', 'namesecondary1_language', 'namesecondary2_text',
       'namesecondary2_language', 'nametertiaryid', 'nametertiary1_text',
       'nametertiary1_language', 'nametertiary2_text',
       'nam

In [18]:
data_waterlinkset.columns

Index(['osid', 'versiondate', 'versionavailablefromdate',
       'versionavailabletodate', 'changetype', 'geometry_length',
       'geometry_evidencedate', 'geometry_updatedate', 'geometry_source',
       'theme', 'description', 'description_evidencedate',
       'description_updatedate', 'description_source', 'name1_text',
       'name1_language', 'name1_evidencedate', 'name1_updatedate',
       'name1_source', 'name2_text', 'name2_language', 'name2_evidencedate',
       'name2_updatedate', 'name2_source', 'geometry'],
      dtype='object')

In [19]:
data_water.address_primarydescription.unique()

array(['Commercial', None, 'Mixed', 'Residential', 'Land'], dtype=object)

In [46]:
from osdatahub import NGD,Extent,PlacesAPI
import os
ngd = NGD(key=os.getenv("OSNDG_API_KEY"),collection="wtr-ntwk-waterlink")
from utils.os_utils import generate_extent_from_artifact
extent = generate_extent_from_artifact(r"exeter_named_area_filtered")
extent

Extent(polygon=Polygon([(inf, inf), (inf, inf), (inf, inf), (inf, inf)]),crs='EPSG:27700')

In [16]:
import joblib
data = joblib.load(r"artifacts/exeter_named_area_filtered.pkl")
data = data.data
data.geometry = data.geometry.to_crs(epsg=27700)
print(data.iloc[0].geometry)

POLYGON ((295173.81 89150.11, 295192.057 89168.355, 295231.155 89233.516, 295325.008 89233.127, 295392.758 89270.006, 295392.766 89301.287, 295406.065 89332.598, 295418.75 89326.9, 295423.95 89330.6, 295429.4 89334.25, 295433.962 89336.987, 295438.35 89339.55, 295445.15 89343.7, 295446.35 89344.75, 295450.205 89348.105, 295454.1 89351.7, 295460.853 89352.325, 295465.038 89353.065, 295473.408 89354.759, 295476.411 89354.865, 295480.433 89355.474, 295486.441 89356.793, 295493.41 89358.569, 295499.959 89358.688, 295511.151 89359.76, 295518.175 89359.522, 295525.319 89360.593, 295530.068 89362.009, 295533.654 89364.046, 295541.512 89367.618, 295544.845 89368.69, 295547.69 89370.899, 295551.037 89374.881, 295554.966 89377.738, 295559.12 89381.376, 295560.562 89383.93, 295562.348 89386.549, 295564.491 89390.121, 295565.761 89393.719, 295566.702 89395.724, 295568.101 89399.042, 295570.17 89405.18, 295571.635 89410.719, 295572.64 89415.216, 295574.016 89419.768, 295575.445 89424.411, 295578.30

In [17]:
extent = Extent.from_bbox(data.iloc[0].geometry.bounds,crs="EPSG:27700")

In [52]:
ngd = NGD(key=os.getenv("OSNDG_API_KEY"),collection="wtr-ntwk-waterlinkset")
output = ngd.query_feature("b67a6a73-5908-4d1e-be31-292304f97ca0")
output["properties"].keys()

dict_keys(['osid', 'theme', 'changetype', 'name1_text', 'name2_text', 'description', 'versiondate', 'name1_source', 'name2_source', 'name1_language', 'name2_language', 'geometry_length', 'geometry_source', 'name1_updatedate', 'name2_updatedate', 'description_source', 'name1_evidencedate', 'name2_evidencedate', 'waterlinkreference', 'geometry_updatedate', 'geometry_evidencedate', 'description_updatedate', 'versionavailabletodate', 'description_evidencedate', 'versionavailablefromdate'])

In [51]:
data_waterlinkset.iloc[0]

osid                                     b67a6a73-5908-4d1e-be31-292304f97ca0
versiondate                                               2025-10-23 00:00:00
versionavailablefromdate                            2025-10-24 00:00:00+00:00
versionavailabletodate                                                    NaT
changetype                                   Modified Geometry And Attributes
geometry_length                                                   4685.892792
geometry_evidencedate                                     2024-04-25 00:00:00
geometry_updatedate                                       2024-04-25 00:00:00
geometry_source                                               Ordnance Survey
theme                                                                   Water
description                                                 Named Watercourse
description_evidencedate                                  2009-01-08 00:00:00
description_updatedate                                    2009-0

In [3]:
data_land = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\lnd_fts_land\lnd_fts_land.gpkg")
data_landpoint = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\lnd_fts_landpoint\lnd_fts_landpoint.gpkg")
data_landform = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\lnd_fts_landform\lnd_fts_landform.gpkg")
data_landformline = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\lnd_fts_landformline\lnd_fts_landformline.gpkg")
data_landformpoint = gpd.read_file(r"C:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\os_ngd_sample\lnd_fts_landformpoint\lnd_fts_landformpoint.gpkg")

In [7]:
print(set(data_landpoint.columns) - set(data_land.columns))

{'name1_source', 'name2_evidencedate', 'name2_language', 'name1_updatedate', 'name2_updatedate', 'description_source', 'name2_text', 'operationalstatus', 'geometry_source', 'name1_evidencedate', 'name1_language', 'name2_source', 'name1_text'}


In [5]:
print(data_land.columns)

Index(['osid', 'toid', 'versiondate', 'versionavailablefromdate',
       'versionavailabletodate', 'firstdigitalcapturedate', 'changetype',
       'geometry_area_m2', 'geometry_evidencedate', 'geometry_updatedate',
       'geometry_capturemethod', 'theme', 'description',
       'description_evidencedate', 'description_updatedate',
       'description_capturemethod', 'oslandcovertiera', 'oslandcovertierb',
       'landform', 'oslandcover_evidencedate', 'oslandcover_updatedate',
       'oslandcover_capturemethod', 'oslandusetiera', 'oslandusetierb',
       'oslanduse_evidencedate', 'oslanduse_updatedate',
       'oslanduse_capturemethod', 'istidal', 'associatedstructure',
       'isobscured', 'physicallevel', 'capturespecification',
       'containingsitecount', 'smallestsite_siteid',
       'smallestsite_landusetiera', 'smallestsite_landusetierb',
       'largestsite_landusetiera', 'largestsite_landusetierb', 'nlud_code',
       'nlud_orderdescription', 'nlud_groupdescription',
       '

In [6]:
print(data_landpoint.columns)

Index(['osid', 'toid', 'versiondate', 'versionavailablefromdate',
       'versionavailabletodate', 'firstdigitalcapturedate', 'changetype',
       'geometry_evidencedate', 'geometry_updatedate', 'geometry_source',
       'theme', 'description', 'description_evidencedate',
       'description_updatedate', 'description_source', 'name1_text',
       'name1_language', 'name1_evidencedate', 'name1_updatedate',
       'name1_source', 'name2_text', 'name2_language', 'name2_evidencedate',
       'name2_updatedate', 'name2_source', 'operationalstatus', 'isobscured',
       'physicallevel', 'capturespecification', 'geometry'],
      dtype='object')


In [10]:
data_landpoint.description.unique()

array(['Non-Coniferous Tree', 'Coniferous Tree'], dtype=object)

In [14]:
print(data_landform.columns)
print(data_landform.name1_text.unique())
print(data_landform.description.unique())

Index(['osid', 'toid', 'versiondate', 'versionavailablefromdate',
       'versionavailabletodate', 'firstdigitalcapturedate', 'changetype',
       'geometry_area', 'geometry_evidencedate', 'geometry_updatedate',
       'geometry_source', 'theme', 'description', 'description_evidencedate',
       'description_updatedate', 'description_source', 'name1_text',
       'name1_language', 'name1_evidencedate', 'name1_updatedate',
       'name1_source', 'name2_text', 'name2_language', 'name2_evidencedate',
       'name2_updatedate', 'name2_source', 'capturespecification', 'geometry'],
      dtype='object')
[None]
['Artificial Slope For Unknown Purpose'
 'Artificial Slope For Other Built Environment'
 'Artificial Slope For Transport' 'Artificial Slope For Flood Controlling'
 'Cliff' 'Artificial Slope For Historic Purpose'
 'Artificial Slope For Flood Or Water Controlling'
 'Artificial Slope For Screening' 'Artificial Slope For Water Controlling']


In [15]:
print(data_landformpoint.columns)
print(data_landformpoint.name1_text.unique())
print(data_landformpoint.description.unique())

Index(['osid', 'toid', 'versiondate', 'versionavailablefromdate',
       'versionavailabletodate', 'firstdigitalcapturedate', 'changetype',
       'geometry_evidencedate', 'geometry_updatedate', 'geometry_source',
       'theme', 'description', 'description_evidencedate',
       'description_updatedate', 'description_source', 'name1_text',
       'name1_language', 'name1_evidencedate', 'name1_updatedate',
       'name1_source', 'name2_text', 'name2_language', 'name2_evidencedate',
       'name2_updatedate', 'name2_source', 'isobscured',
       'capturespecification', 'geometry'],
      dtype='object')
[]
[]


In [17]:
print(data_landformline.columns)
print(data_landformline.description.unique())

Index(['osid', 'toid', 'versiondate', 'versionavailablefromdate',
       'versionavailabletodate', 'firstdigitalcapturedate', 'changetype',
       'geometry_length', 'geometry_evidencedate', 'geometry_updatedate',
       'geometry_source', 'theme', 'description', 'description_evidencedate',
       'description_updatedate', 'description_source', 'isobscured',
       'capturespecification', 'geometry'],
      dtype='object')
['Bottom Or Side Of Slope' 'Top Of Slope' 'Cliff Edge'
 'Base Or Side Of Cliff']


### Added Water Features and Water Network Lets try it out

In [1]:
# First the imports and the keys
import warnings
warnings.filterwarnings("ignore")
from utils.keys import set_api_keys
import os
from utils.tool_definitions import send_message_definitions
from pydantic import BaseModel,Field
from utils.tool_definitions import *
from a2a.Agent import Agent
from utils.card_templates import *
from a2a.Artifact import Artifact
import pandas as pd
import joblib
# We set the api keys as earlier
from utils.tools import *
from utils.prompt_templates import *
set_api_keys()

In here
Openai and ngd key set successfully


In [2]:
# Ok Step 1 : Let us initialise our agents tools and utilities that the agents will use.


#coding agent
tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]

coding_agent = Agent(coding_agent_details,
                     "gpt-4.1",
                     None,
                     tools,
                     tool_definitions,
                     additional_args={"parallel_tool_calls":False},
                     available_agents=None,
                     system_instruction=generic_coding_agent_template)


#plotting agent
tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]
plotting_agent = Agent(plotting_agent_card,
                     "gpt-4.1",
                     None,
                     tools,
                     tool_definitions,
                     additional_args={"parallel_tool_calls":False},
                     available_agents=None,
                     system_instruction=plotting_agent_template)



# Address agent
address_agent_cards =[coding_agent_details.model_dump()]
address_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
address_system_prompt = places_prompt + f"\n <AVAILABLE AGENTS> {address_agent_cards} <AVAILABLE AGENTS>"

address_agent = Agent(agent_details=address_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=address_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=address_system_prompt)


# Buildings agent
buildings_agent_cards =[coding_agent_details.model_dump()]
buildings_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
buildings_system_prompt = buildings_prompt + f"\n <AVAILABLE AGENTS> {buildings_agent_cards} <AVAILABLE AGENTS>"

buildings_agent = Agent(agent_details=building_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=buildings_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=buildings_system_prompt)



# Named Area agent
named_area_agent_cards =[coding_agent_details.model_dump()]
named_area_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
named_area_system_prompt = named_area_prompt + f"\n <AVAILABLE AGENTS> {named_area_agent_cards} <AVAILABLE AGENTS>"

named_area_agent = Agent(agent_details=named_area_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=named_area_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=named_area_system_prompt)

# Water Features agent
water_features_agent_cards =[coding_agent_details.model_dump()]
water_features_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
water_features_system_prompt = water_features_prompt + f"\n <AVAILABLE AGENTS> {water_features_agent_cards} <AVAILABLE AGENTS>"
water_features_agent = Agent(agent_details=water_features_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=water_features_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=water_features_system_prompt)

# Water Network agent
water_network_agent_cards =[coding_agent_details.model_dump()]
water_network_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
water_network_system_prompt = water_network_prompt + f"\n <AVAILABLE AGENTS> {water_network_agent_cards} <AVAILABLE AGENTS>"
water_network_agent = Agent(agent_details=water_network_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=water_network_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=water_network_system_prompt)


# planning agent
planning_agent = Agent(agent_details=planning_agent_card,
                       llm_name = "gpt-4o-mini", schema=None,
                       tools=None,
                       tool_definitions=None,
                       additional_args=None,
                       available_agents=None,
                       artifacts_req=None,
                       system_instruction=planning_agent_prompt)

# host agent
host_agent_cards = [planning_agent_card.model_dump(),address_agent_card.model_dump(), named_area_agent_card.model_dump(),building_agent_card.model_dump(),water_features_agent_card.model_dump(),
                    water_network_agent_card.model_dump(),plotting_agent_card.model_dump()]
host_agent_tools = {"send_message":send_message,'generate_metadata_for_all_artifacts':generate_metadata_for_all_artifacts}
host_tools_definitions = [send_message_definitions,metadata_all_artifacts]
host_system_message = host_prompt_template_for_os_version_2 + f"\n <AVAILABLE AGENTS> {host_agent_cards} <AVAILABLE AGENTS>"

host_agent = Agent(agent_details=host_agent_card,
                   llm_name="gpt-4.1",schema=None,
                   tools=host_agent_tools,
                   tool_definitions=host_tools_definitions,
                   additional_args={"parallel_tool_calls":False},
                   available_agents=[planning_agent,address_agent,named_area_agent,buildings_agent,water_features_agent,water_network_agent,plotting_agent],
                   artifacts_req=None,
                   system_instruction=host_system_message)


In [8]:
messages = [{"role":"user","content":"Exeter banks farther than a mile to any police station, show exeter area, banks, mile circle around police station and banks farther than a mile from police stations"}]
host_agent.run_agent(messages)

Calling tool send_message with args : {'target': 'planning_agent', 'task_description': 'Plan the steps to solve the query: Show Exeter area, banks, a mile circle around police stations, and banks farther than a mile from police stations.'}
Messages sent to agent planning_agent [{'role': 'user', 'content': 'Plan the steps to solve the query: Show Exeter area, banks, a mile circle around police stations, and banks farther than a mile from police stations.'}]
Calling tool send_message with args : {'target': 'named_area', 'task_description': 'Find the area polygon for Exeter.'}
Messages sent to agent named_area [{'role': 'user', 'content': 'Find the area polygon for Exeter.'}]
Calling tool call_os_ngd with args : {'filters': ['Exeter'], 'bbox': None, 'polygon_or_point': True, 'street_address': None, 'filename': 'exeter_area_polygon'}
NGD query result {'name': 'exeter_area_polygon', 'description': 'A geopandas dataframe containing named area data with bbox applied as per user request.', 'co

('Here’s what was done to answer your query:\n\n1. The Exeter area was identified and mapped.\n2. All police stations within Exeter were found and plotted.\n3. A 1-mile circle (buffer) was drawn around each police station.\n4. All banks in Exeter were found and plotted.\n5. Banks that are farther than a mile from any police station were highlighted.\n\nThe map shows:\n- The Exeter area boundary\n- Police stations and their 1-mile buffer zones\n- All banks in Exeter\n- Banks that are farther than a mile from any police station (highlighted)\n\nExample of banks farther than a mile from any police station:\n- FRASER AND WHEELER LTD, 35 Cowick Street, Exeter\n- BARCLAYS, Alphinbrook Road, Marsh Barton, Exeter\n- LLOYDS TSB BANK PLC, 13 St Thomas Centre, Exeter\n- NATWEST, Stocker Road, Exeter\n- LLOYDS TSB BANK PLC, Main...\n\nYou can view all these layers and spatial relationships in the artifact: exeter_police_banks_map (exeter_police_banks_map.html). If you need the map file or more det

### Added Land, lets see if it works or not

In [1]:
# First the imports and the keys
import warnings
warnings.filterwarnings("ignore")
from utils.keys import set_api_keys
import os
from utils.tool_definitions import send_message_definitions
from pydantic import BaseModel,Field
from utils.tool_definitions import *
from a2a.Agent import Agent
from utils.card_templates import *
from a2a.Artifact import Artifact
import pandas as pd
import joblib
# We set the api keys as earlier
from utils.tools import *
from utils.prompt_templates import *
set_api_keys()

In here
Openai and ngd key set successfully


In [2]:
# Ok Step 1 : Let us initialise our agents tools and utilities that the agents will use.


#coding agent
tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]

coding_agent = Agent(coding_agent_details,
                     "gpt-4.1",
                     None,
                     tools,
                     tool_definitions,
                     additional_args={"parallel_tool_calls":False},
                     available_agents=None,
                     system_instruction=generic_coding_agent_template)


#plotting agent
tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]
plotting_agent = Agent(plotting_agent_card,
                     "gpt-4.1",
                     None,
                     tools,
                     tool_definitions,
                     additional_args={"parallel_tool_calls":False},
                     available_agents=None,
                     system_instruction=plotting_agent_template)



# Address agent
address_agent_cards =[coding_agent_details.model_dump()]
address_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
address_system_prompt = places_prompt + f"\n <AVAILABLE AGENTS> {address_agent_cards} <AVAILABLE AGENTS>"

address_agent = Agent(agent_details=address_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=address_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=address_system_prompt)


# Buildings agent
buildings_agent_cards =[coding_agent_details.model_dump()]
buildings_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
buildings_system_prompt = buildings_prompt + f"\n <AVAILABLE AGENTS> {buildings_agent_cards} <AVAILABLE AGENTS>"

buildings_agent = Agent(agent_details=building_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=buildings_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=buildings_system_prompt)



# Named Area agent
named_area_agent_cards =[coding_agent_details.model_dump()]
named_area_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
named_area_system_prompt = named_area_prompt + f"\n <AVAILABLE AGENTS> {named_area_agent_cards} <AVAILABLE AGENTS>"

named_area_agent = Agent(agent_details=named_area_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=named_area_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=named_area_system_prompt)

# Water Features agent
water_features_agent_cards =[coding_agent_details.model_dump()]
water_features_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
water_features_system_prompt = water_features_prompt + f"\n <AVAILABLE AGENTS> {water_features_agent_cards} <AVAILABLE AGENTS>"
water_features_agent = Agent(agent_details=water_features_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=water_features_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=water_features_system_prompt)

# Water Network agent
water_network_agent_cards =[coding_agent_details.model_dump()]
water_network_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
water_network_system_prompt = water_network_prompt + f"\n <AVAILABLE AGENTS> {water_network_agent_cards} <AVAILABLE AGENTS>"
water_network_agent = Agent(agent_details=water_network_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=water_network_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=water_network_system_prompt)

# land agent
land_features_agent_cards = [coding_agent_details.model_dump()]
land_features_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
land_features_system_prompt = land_features_prompt + f"\n <AVAILABLE AGENTS> {land_features_agent_cards} <AVAILABLE AGENTS>"
land_features_agent = Agent(agent_details=land_features_agent_card,
                        llm_name="gpt-4o",
                        schema=None,
                        tools=land_features_agent_tools,
                        tool_definitions=[send_message_definitions,os_ngd_tool_description],
                        additional_args={"parallel_tool_calls":False},
                         available_agents=[coding_agent],
                         artifacts_req=None,
                         system_instruction=land_features_system_prompt)


# planning agent
planning_agent = Agent(agent_details=planning_agent_card,
                       llm_name = "gpt-4o-mini", schema=None,
                       tools=None,
                       tool_definitions=None,
                       additional_args=None,
                       available_agents=None,
                       artifacts_req=None,
                       system_instruction=planning_agent_prompt)

# host agent
host_agent_cards = [planning_agent_card.model_dump(),address_agent_card.model_dump(), named_area_agent_card.model_dump(),building_agent_card.model_dump(),water_features_agent_card.model_dump(),
                    water_network_agent_card.model_dump(),land_features_agent_card.model_dump(),plotting_agent_card.model_dump()]
host_agent_tools = {"send_message":send_message,'generate_metadata_for_all_artifacts':generate_metadata_for_all_artifacts}
host_tools_definitions = [send_message_definitions,metadata_all_artifacts]
host_system_message = host_prompt_template_for_os_version_2 + f"\n <AVAILABLE AGENTS> {host_agent_cards} <AVAILABLE AGENTS>"

host_agent = Agent(agent_details=host_agent_card,
                   llm_name="gpt-4.1",schema=None,
                   tools=host_agent_tools,
                   tool_definitions=host_tools_definitions,
                   additional_args={"parallel_tool_calls":False},
                   available_agents=[planning_agent,address_agent,named_area_agent,buildings_agent,water_features_agent,water_network_agent,land_features_agent,plotting_agent],
                   artifacts_req=None,
                   system_instruction=host_system_message)


In [3]:
messages = [{"role":"user","content":"Find woodlands in Exeter"}]
host_agent.run_agent(messages)

Calling tool send_message with args : {'target': 'planning_agent', 'task_description': 'The user wants to find woodlands in Exeter. Provide the sequence of steps and which agents/artifacts to use.'}
Messages sent to agent planning_agent [{'role': 'user', 'content': 'The user wants to find woodlands in Exeter. Provide the sequence of steps and which agents/artifacts to use.'}]
Calling tool send_message with args : {'target': 'named_area', 'task_description': 'Find the area polygon for Exeter.'}
Messages sent to agent named_area [{'role': 'user', 'content': 'Find the area polygon for Exeter.'}]
Calling tool call_os_ngd with args : {'filters': ['Exeter'], 'bbox': None, 'polygon_or_point': True, 'street_address': None, 'filename': 'exeter_polygon_data'}
NGD query result {'name': 'exeter_polygon_data', 'description': 'A geopandas dataframe containing named area data with bbox applied as per user request.', 'count': 98}
Calling tool send_message with args : {'target': 'data_analysis_agent', 

('I have found and mapped the woodlands within Exeter:\n\n- The Exeter area was identified and its boundary polygon was used.\n- 490 woodland polygons (such as coniferous trees) were found within Exeter.\n- A map has been generated showing all woodland areas within the Exeter boundary, with the boundary outlined in blue and woodlands filled in green.\n\nIf you would like to view or download the map, let me know!',
 [<a2a.Artifact.Artifact at 0x247383bd450>])

### Here we need to figure out adding human as a node in the loop and then trying to ask human to solve possible ambiguity
#### Several ways to solve this:
* We can make the humans as node and then when the LLM detects ambiguity it can ask the human
* Questions are 
  * Are there specific points of ambiguity like is it only restricted to the conditions (near, close by)
  * Which agents will have the power to raise ambiguity, host or ngd agents too
  *  Test to find out how good are LLMs at not only detecting uncertainty but detecting it consistently
  
### Test 1: lets make human as a node and then let us give the host agent the capability to communicate with the human.

In [1]:
# First the imports and the keys
import warnings
warnings.filterwarnings("ignore")
from utils.keys import set_api_keys
import os
from utils.tool_definitions import send_message_definitions
from pydantic import BaseModel,Field
from utils.tool_definitions import *
from a2a.Agent import Agent
from a2a.Human import Human
from utils.card_templates import *
from a2a.Artifact import Artifact
import pandas as pd
import joblib
# We set the api keys as earlier
from utils.tools import *
from utils.prompt_templates import *
set_api_keys()

In here
Openai and ngd key set successfully


In [2]:
# Ok Step 1 : Let us initialise our agents tools and utilities that the agents will use.


#coding agent
tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]

coding_agent = Agent(coding_agent_details,
                     "gpt-4.1",
                     None,
                     tools,
                     tool_definitions,
                     additional_args={"parallel_tool_calls":False},
                     available_agents=None,
                     system_instruction=generic_coding_agent_template)


#plotting agent
tools = {"code_executor":code_executor,"generate_metadata_for_artifacts":generate_metadata_for_artifacts}
tool_definitions = [code_executor_definition, code_metadata_generator_definition]
plotting_agent = Agent(plotting_agent_card,
                     "gpt-4.1",
                     None,
                     tools,
                     tool_definitions,
                     additional_args={"parallel_tool_calls":False},
                     available_agents=None,
                     system_instruction=plotting_agent_template)



# Address agent
address_agent_cards =[coding_agent_details.model_dump()]
address_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
address_system_prompt = places_prompt + f"\n <AVAILABLE AGENTS> {address_agent_cards} <AVAILABLE AGENTS>"

address_agent = Agent(agent_details=address_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=address_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=address_system_prompt)


# Buildings agent
buildings_agent_cards =[coding_agent_details.model_dump()]
buildings_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
buildings_system_prompt = buildings_prompt + f"\n <AVAILABLE AGENTS> {buildings_agent_cards} <AVAILABLE AGENTS>"

buildings_agent = Agent(agent_details=building_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=buildings_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=buildings_system_prompt)



# Named Area agent
named_area_agent_cards =[coding_agent_details.model_dump()]
named_area_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
named_area_system_prompt = named_area_prompt + f"\n <AVAILABLE AGENTS> {named_area_agent_cards} <AVAILABLE AGENTS>"

named_area_agent = Agent(agent_details=named_area_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=named_area_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=named_area_system_prompt)

# Water Features agent
water_features_agent_cards =[coding_agent_details.model_dump()]
water_features_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
water_features_system_prompt = water_features_prompt + f"\n <AVAILABLE AGENTS> {water_features_agent_cards} <AVAILABLE AGENTS>"
water_features_agent = Agent(agent_details=water_features_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=water_features_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=water_features_system_prompt)

# Water Network agent
water_network_agent_cards =[coding_agent_details.model_dump()]
water_network_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
water_network_system_prompt = water_network_prompt + f"\n <AVAILABLE AGENTS> {water_network_agent_cards} <AVAILABLE AGENTS>"
water_network_agent = Agent(agent_details=water_network_agent_card,
                      llm_name="gpt-4o",
                      schema=None,
                      tools=water_network_agent_tools,
                      tool_definitions=[send_message_definitions,os_ngd_tool_description],
                      additional_args={"parallel_tool_calls":False},
                     available_agents=[coding_agent],
                     artifacts_req=None,
                     system_instruction=water_network_system_prompt)

# land agent
land_features_agent_cards = [coding_agent_details.model_dump()]
land_features_agent_tools = {"send_message":send_message,"call_os_ngd":call_os_ngd}
land_features_system_prompt = land_features_prompt + f"\n <AVAILABLE AGENTS> {land_features_agent_cards} <AVAILABLE AGENTS>"
land_features_agent = Agent(agent_details=land_features_agent_card,
                        llm_name="gpt-4o",
                        schema=None,
                        tools=land_features_agent_tools,
                        tool_definitions=[send_message_definitions,os_ngd_tool_description],
                        additional_args={"parallel_tool_calls":False},
                         available_agents=[coding_agent],
                         artifacts_req=None,
                         system_instruction=land_features_system_prompt)


# planning agent
planning_agent = Agent(agent_details=planning_agent_card,
                       llm_name = "gpt-4o-mini", schema=None,
                       tools=None,
                       tool_definitions=None,
                       additional_args=None,
                       available_agents=None,
                       artifacts_req=None,
                       system_instruction=planning_agent_prompt)

# human agent
human_agent = Human(human_agent_card)

# host agent
host_agent_cards = [planning_agent_card.model_dump(),address_agent_card.model_dump(), named_area_agent_card.model_dump(),building_agent_card.model_dump(),water_features_agent_card.model_dump(),
                    water_network_agent_card.model_dump(),land_features_agent_card.model_dump(),plotting_agent_card.model_dump(),human_agent_card.model_dump()]
host_agent_tools = {"send_message":send_message,'generate_metadata_for_all_artifacts':generate_metadata_for_all_artifacts}
host_tools_definitions = [send_message_definitions,metadata_all_artifacts]
host_system_message = host_prompt_template_for_os_version_2 + f"\n <AVAILABLE AGENTS> {host_agent_cards} <AVAILABLE AGENTS>"

host_agent = Agent(agent_details=host_agent_card,
                   llm_name="gpt-4.1",schema=None,
                   tools=host_agent_tools,
                   tool_definitions=host_tools_definitions,
                   additional_args={"parallel_tool_calls":False},
                   available_agents=[planning_agent,address_agent,named_area_agent,buildings_agent,water_features_agent,water_network_agent,land_features_agent,plotting_agent,human_agent],
                   artifacts_req=None,
                   system_instruction=host_system_message)


In [3]:
human_send_message(message="yes please go ahead and make the map plot, do not give me the code but execute the code",target_agent=[host_agent])

Messages sent to agent host_agent [{'role': 'assistant', 'content': 'All university points have been plotted with a clearly visible 5km buffer (red dashed outline, light blue fill) around each point. Additionally, all restaurant/food place points that fall within these buffers are highlighted in red. A legend is included to distinguish university points, the buffer, and restaurants within the buffer.\n\nSummary:\n- University points are marked in blue.\n- 5km buffers are shown as red dashed outlines with light blue fill.\n- Restaurants/food places within the buffer are marked in red.\n- A legend is provided for clarity.\n\nIf you would like to view the map or need more details about the restaurants within the buffer, let me know!'}, {'role': 'user', 'content': 'For each university points plot the 5km buffer and then show all the places to eat. Currently it is not that'}, {'role': 'assistant', 'content': 'All university points will be plotted, each with a clearly visible 5km buffer (red

['The map has been successfully generated:\n\n- All university points are plotted, each with a clearly visible 5km buffer (red dashed outline, light blue fill).\n- All places to eat are shown as orange markers, regardless of whether they fall inside or outside the buffer.\n- A legend is included to distinguish university points, buffers, and food places.\n\nYou can view the map in the artifact: university_points_buffers_food_places_map (file: university_points_buffers_food_places_map.html).\n\nIf you need a summary of the food places within the buffer or any further customizations, let me know!Addtionally some data artifacts have been generated with names  [\'university_points_buffers_food_places_map\'] and \n descriptions ["A folium map showing all university points from \'filtered_university_of_exeter_results\' with 5km buffers (red dashed outline, light blue fill) and all food places from \'exeter_food_places\' (orange markers). Includes a legend for clarity."]',

### Optmization stages : This is the stage where we will improve performance. Checklist
* Plotting function prompt change so that it does not return code
* Doing something about artifact names as they are point of ambiguity when host hallucinates
* Adding an initialisation script to prevent copy pasting the code again and again
* Adding specifications to the description generated when artifacts are generated by tool itself
* Ambiguity criteria setting in prompt or adding another agent (need to design the seq here)

### Open Source Integration: after ensure that the above quality improvements are complete we can set up Openrouter to investigate open source suitability
* Try Meta LLama 4 and Qwen (points of open source integration to be decided)
* May even try claude for coding 

### Finally UI improvements:
* Thinking of a split screen UI with graph visualization on left and chat on right

### Lastly update README.md

In [ ]:
# Written a fairly complex initalisation script for the Ordnance Survey agents and tools.
# It can make the framework using the config file in agent_frameworks folder. Lets test it out.
# The current config file does not have human will integrate human node in a different file

import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
set_api_keys()

In here
Openai and ngd key set successfully


In [2]:
config = None
with open(r"agent_frameworks\agent_config.json","rb") as file:
    config = json.load(file)

# Now that things are initialised
agent_archiecture = OSAgentsInitializer(config).initialize_all_agents()

Agent host_agent Initialised
Agent planning_agent Initialised
Agent coding_agent Initialised
Agent plotting_agent Initialised
Agent address_agent Initialised
Agent named_area_agent Initialised
Agent buildings_agent Initialised
Agent water_features_agent Initialised
Agent water_network_agent Initialised
Agent land_features_agent Initialised


In [ ]:
# We can run a query in this way.
messages = [{"role":"user","content":"Find woodlands in Exeter"}]
agent_archiecture["host_agent"].run_agent(messages)


Calling tool send_message with args : {'target': 'planning_agent', 'task_description': 'The user wants to find woodlands in Exeter. Provide the sequence of steps and which agents to use for this query.'}
Messages sent to agent planning_agent [{'role': 'user', 'content': 'The user wants to find woodlands in Exeter. Provide the sequence of steps and which agents to use for this query.'}]
Output from agent planning_agent : <Internal Thoughts> The area is Exeter, so there will be 1 entry for Exeter. The entity to search for is woodlands, and we need to find as many entries as possible within Exeter. <Thoughts>

output steps: ["Find Exeter 1 area", "Search for woodlands in Exeter as many search results"] 

Agents to use: Geospatial search agent for locating woodlands, and a data retrieval agent for gathering information on the woodlands found.
Calling tool send_message with args : {'target': 'named_area', 'task_description': 'Find the area polygon for Exeter.'}
Messages sent to agent named_

('I have found and mapped the woodlands within Exeter:\n\n- 509 woodland areas were identified, including types such as "Scattered Non-Coniferous Trees," "Scattered Mixed Trees," and "Mixed Trees."\n- These woodlands are all located within the official Exeter area boundary.\n\nA map has been generated showing all woodland polygons within Exeter, with the area boundary outlined in blue and woodlands highlighted in green.\n\nIf you would like to view or download the map, let me know! If you need more details about specific woodlands, I can provide those as well.',
 [<a2a.Artifact.Artifact at 0x21e3342da10>])